# normalization with uMAIA, making two sections comparable

In notebook N1 you pulled raw MALDI images from METASPACE, and in N2 you turned those peaks into named
lipids. You now hold, for each of our two coronal sections, a table of pixels by lipids. One section is
a control female brain, the other a pregnant female brain, cut at approximately the *same* coronal plane. One key question: which lipids change in the brain
during pregnancy (or disease, for example), and where? Before we can answer it, we have to make the two sections honestly
comparable, because they were acquired on different days, on different slides, and the instrument does
not give the same number twice for the same amount of lipid.

That is what this notebook is about. We meet the **batch effect**, the technical drift that separates
two acquisitions; we look at the empirical structure of MALDI data that makes it correctable; we read
the **uMAIA** model that exploits that structure; and then we actually run uMAIA on our two sections,
watch the slide-to-slide offset shrink, and confirm that the white-matter sphingolipid signal lines up
across sections afterwards. We unroll uMAIA's correction by hand so it is not a black box.

IMPORTANT: of course batch effect correction in a 1 vs 1 setting is ill-defined. any shift between the two sections here could be due to either technical batch effects or biological variation, the latter inter-individual or pregnancy-driven. In fact, this kind of analysis is properly done at atlas scale, where several sections x multiple individuals are measured (eg 6 sections x 4 individuals x 2 conditions). To keep things easy, we'll work with 1 vs 1 sections only here, but beware of this intentional mistake.

In [ ]:
# --- this notebook RUNS uMAIA for real, on the cajal-umaia kernel ---
# uMAIA's fit is JAX/NumPyro. To avoid complications and maybe run on your own machine, we avoid GPUs, so we pin JAX to the CPU.
# This MUST happen before anything imports jax (uMAIA does, on import), so it is the
# very first line of the notebook.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

# the scientific-Python stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import stats                      # the normal CDF for the histogram-matching transform
from scipy.interpolate import interp1d

# uMAIA itself (only this kernel has it installed)
import uMAIA

# the course helper package + shared plotting style
import sys; sys.path.insert(0, os.path.abspath("../../src"))
from cajal_lipidomics import analysis
from cajal_lipidomics.plotting import FS
from cajal_lipidomics.style import set_style, CMAP_SEQUENTIAL
set_style()

DATA = os.path.abspath("../../data")
RNG_SEED = 42
import jax
print(f"ready. numpy {np.__version__} | anndata {ad.__version__} | JAX on {jax.devices()[0].platform}")

**check.** You should see `ready. numpy ... | anndata ... | JAX on cpu` and no red error. This
notebook runs on the **`cajal-umaia`** kernel, not `cajal-lipidomics`: it is the only one with JAX,
NumPyro and uMAIA installed. If you get `ModuleNotFoundError: No module named 'uMAIA'`, you are on the
wrong kernel; pick `cajal-umaia` in the top-right kernel picker and rerun. The `JAX on cpu` at the end
confirms we pinned JAX to the CPU, because the course cluster has no GPU.

## the two sections, straight from N2

We pick the chain up exactly where N2 left it: `data/derived/02_annotated.h5ad`. This is *your* output
from the previous notebook, the annotated substrate, one AnnData holding both sections stacked together.
`adata.X` is the matrix of pixels by lipids, still on its **raw** intensity scale (nothing has been
normalized yet), `adata.var` carries the lipid names you assigned, and `adata.obs` carries the per-pixel
metadata: which condition, which section, and where each pixel sits in the Allen brain atlas.

🔬 **TASK.** Load it and read off the shape.

In [ ]:
# the annotated two-section substrate from N2 (pixels x lipids), raw intensities
adata = ad.read_h5ad(os.path.join(DATA, "derived", "02_annotated.h5ad"))

n_pixels, n_lipids = adata.shape
print(f"{n_pixels:,} pixels  x  {n_lipids} lipids")
print()
print("pixels per condition:")
print(adata.obs["Condition"].value_counts())
print()
print("a few lipid names:", [l for l in adata.var["lipid"] if not l.startswith("ion_")][:6])
print("X range:", float(np.asarray(adata.X).min()), "to", float(np.asarray(adata.X).max()))

**check.** 189,011 pixels and 104 lipids, split into `naive` (the control female, about 89k
pixels) and `pregnant` (about 100k pixels). The lipid names read as a class plus a chain summary:
`HexCer 42:2` is a hexosylceramide with 42 acyl carbons and 2 double bonds, a sphingolipid that marks
myelin; we met that naming in N2. `X` runs from 0 to 1 because the ion images arrive from METASPACE as
8-bit intensity images, scaled into [0, 1] per image. That is the raw scale we start from. The
quantization is fine for everything below.

## what a batch effect is, in one picture

MALDI-MSI acquires one mass spectrum per pixel. A whole tissue section is one **acquisition**: one
laser run, one matrix coating, one instrument session. Between acquisitions several non-biological
things drift:

- **matrix crystallization**: the light-absorbing matrix is sprayed onto the slide and crystallizes;
  thicker or finer crystals desorb more or less material.
- **laser energy**: the pulse energy is not identical shot to shot or day to day.
- **detector gain and ionization efficiency**: the electronics and the ionization yield wander.
- **sample handling**: thaw-mounting, storage time, humidity.

None of these has anything to do with biology, yet all of them move the measured intensity of a given
lipid up or down by a roughly section-wide factor. That is the **batch effect**: a technical shift that affects one whole acquisition relative to another. If we compared our control
and pregnant sections without correcting it, part of every difference we found would just be "the
pregnant slide ran hotter", and we would have no way to tell that apart from real biology.

The saving grace is the experimental design (WITH THE CAVEAT IN THE INTRO). The two sections are cut at the same coronal plane, so the
**anatomy is matched**: the same regions, in the same places, in both. When the anatomy is held fixed,
any blanket intensity difference between the slides is suspicious, and that is exactly the handle a
normalizer can grab. Let us look at the two sections first, painted by Allen region, to see how well
the anatomy lines up.

In [ ]:
# plot both sections coloured by Allen anatomical region (the provided per-pixel colours)
secs = ["naive", "pregnant"]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, cond in zip(axes, secs):
    m = (adata.obs["Condition"] == cond).to_numpy()
    ax.scatter(adata.obs.loc[m, "zccf"], -adata.obs.loc[m, "yccf"],
               c=adata.obs.loc[m, "allencolor"], s=2, rasterized=True)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(f"{cond}  ({m.sum():,} pixels)", fontsize=FS["m"])
fig.suptitle("the two coronal sections, painted by Allen region", fontsize=FS["l"])
plt.tight_layout(); plt.show()

**check.** Two coronal brain slices of nearly identical shape, painted in the Allen colour scheme.
Same plane, same anatomy, different mouse and different acquisition. That is the ideal setup for spotting
a batch effect: when the *anatomy* is matched, any blanket intensity difference between the slides is
technical, not biological (with the caveat above: here it is confounded, it is technical if we decide the two sections are replicates of the same biological archetype).

❓ **QUESTION.** A batch effect shifts a whole section's intensity scale at once. A real biological
difference changes specific lipids in specific places. Keep that contrast in mind: normalization tries
to remove the first without touching the second. How might a method possibly tell them apart from the
data alone? The next section gives uMAIA's answer.

## the key empirical fact: per-molecule histograms are bimodal in log space

uMAIA does not just rescale each image by a constant. It leans on a specific structure that real MALDI
data has. Take one lipid, collect its intensity across all the pixels of one section, and plot the
histogram **in log space**. You typically see *two humps*, not one:

- a low **background mode**: pixels where this lipid is essentially absent, so you are measuring matrix
  noise and baseline. These pixels pile up at a low, roughly fixed value.
- a higher **foreground mode**: pixels where the lipid is genuinely present, at a level set by the
  tissue's biology.

Why log space? Because intensities span orders of magnitude and the interesting structure is
multiplicative. On a linear axis everything crushes against zero and the two humps merge into one spike.
On a log axis they separate and you can see them. Let us look at one myelin sphingolipid, `HexCer 42:2`,
in the control section.

In [ ]:
# pick one lipid and one section; collect its intensities across all pixels
lipid = "HexCer 42:2"
j = list(adata.var["lipid"]).index(lipid)
X = np.asarray(adata.X)                        # pixels x lipids, raw scale
eps = 2e-4                                      # uMAIA's log offset (mirrors build_umaia_input.py)

m_ctrl = (adata.obs["Condition"] == "naive").to_numpy()
vals = X[m_ctrl, j]                             # this lipid's intensity in every control pixel
log_vals = np.log(np.clip(vals, 0, None) + eps)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(vals, bins=80, color="steelblue")
axes[0].set_title("linear axis: structure is invisible", fontsize=FS["m"])
axes[0].set_xlabel(f"{lipid} raw intensity"); axes[0].set_ylabel("pixels")
axes[1].hist(log_vals, bins=80, color="steelblue")
axes[1].set_title("log axis: two modes appear", fontsize=FS["m"])
axes[1].set_xlabel(f"log({lipid} + eps)")
floor = np.log(eps) + 0.3
print(f"fraction of control pixels at the background floor: {(log_vals < floor).mean():.0%}")
plt.tight_layout(); plt.show()

**check.** On the linear axis everything crushes against zero. On the log axis you see the
structure: a sharp spike at the very bottom (the background floor, where the lipid is absent, about a
third of pixels) and a broad hump to the right (the foreground, the white-matter pixels where this
myelin lipid is genuinely abundant). Not every lipid is this clearly bimodal, but many are, and that is
the structure uMAIA exploits.

The crucial asymmetry, established in the uMAIA paper: **the batch effect acts mostly on the foreground
mode.** The background mode is matrix noise and stays roughly put from slide to slide; it is a stable
anchor. The foreground mode is real signal, and that is where the technical gain wanders. So the
correction we want is not "shift everything by a constant"; it is "hold the background anchor, and slide
the foreground mode back into register". Keep that picture: anchor low, drift high.

🔬 **TASK.** Now overlay the *same lipid* from the two sections, in log space, to see whether their
foreground modes sit at the same place. This is the batch effect, if it is there.

In [ ]:
# overlay the log histogram of the SAME lipid in the two sections
# 🔬 your code here


**check.** Replot the above histogram for several lipids and eyeball it. The two background floors sit on top of each other (the anchor), but the two foreground
humps do *not* line up for all lipids; the dashed lines mark each section's foreground median and they are
visibly offset, on the order of 0.2 to 0.4 log units. That gap is the thing we have to worry about. Some
of it might be real (pregnancy could genuinely change this lipid) and some is technical drift, and on
this raw data the two are tangled together. Normalization's job is to remove the technical part while
disturbing the biological part as little as possible. To see how, we need the model.

## the uMAIA model: a mixture per (lipid, section), with a rank-1 batch term

uMAIA fits, for every (lipid, section) pair, a **two-component Gaussian mixture in log space**: one
Gaussian for the background mode, one for the foreground mode. The clever part is how the foreground
mode's position is parameterized. Write `x` for a pixel's log-intensity of compound `c` in acquisition
(section) `a`. The generative model, read off uMAIA's `_model.py` on its default path
(`delta_v_dist='gaussian'`, the branch `normalize()` actually passes), is:

```
# --- per-lipid parameters: one value per compound c, SHARED across sections ---
delta_v_c ~ Gamma(5.0, 2.0)           # prior mean 2.5: the typical fg-minus-bg gap on the log scale
delta_c   ~ Normal(delta_v_c, 0.25)   # this lipid's mode gap, tight around delta_v_c
locs_c    ~ Normal(m0, 1)             # background mean = the anchor, ONE per lipid (not per section)
sigma0_c  ~ Uniform(0.05, 0.5)        # background sd (uMAIA stores this under the param name `scale1`)
lambda_c  ~ Uniform(-2, 2)            # how susceptible THIS lipid is to batch drift
sigma_v_c ~ Exponential(.)            # per-lipid piece of the foreground sd

# --- per-section parameters: one value per acquisition a ---
gamma_a   ~ Uniform(-2, 2)            # how strong the drift is on THIS slide
sigma_s_a ~ Exponential(.)            # per-slide piece of the foreground sd

# --- per (lipid, section) slack on the batch shift ---
error_ac  ~ Uniform(-0.5, 0.5)        # small wiggle that soaks up departures from the strict rank-1 shift

# --- assembled per (lipid, section) ---
mu0_ac    = locs_c                                              # background mean (the anchor), per lipid
mu1_ac    = locs_c + gamma_a * lambda_c + delta_c + error_ac    # foreground mean
sigma1_ac = sigma_s_a + sigma_v_c                              # foreground sd, built ADDITIVELY
pi_ca     ~ Dirichlet(0.5, 0.5)       # mixture weights: how much background vs foreground
z_p       ~ Bernoulli(pi_ca)          # per-pixel latent label: is this pixel bg or fg?
x_p       ~ z * Normal(mu1_ac, sigma1_ac) + (1 - z) * Normal(mu0_ac, sigma0_c)
```

A note on what is per-lipid versus per-section, because it is easy to get wrong: in the code `locs`
(the background anchor) is sampled **per lipid only**, one vector over compounds, not one per (lipid,
section). The parameters that carry a section index are `gamma_a`, `sigma_s_a`, and the small per-(lipid, section) slack `error_ac`. So the background
mean of a given lipid is the same value the model places under every section's background hump, which
is exactly what makes it an anchor.

Read the foreground-mean line slowly, because it is the heart of the method:

$$\mu^{1}_{ac} \;=\; \underbrace{\text{locs}_{c}}_{\text{background anchor (per lipid)}} \;+\; \underbrace{\gamma_a \cdot \lambda_c}_{\text{batch shift}} \;+\; \underbrace{\delta_c}_{\text{biological mode gap}}$$

The code carries one term the picture above leaves out, a small per-(lipid, section) slack
`error_ac ~ Uniform(-0.5, 0.5)` added to the same foreground mean. It absorbs the bit of drift the
strict rank-1 product cannot, and the correction removes it together with `gamma_a * lambda_c`, so it
changes nothing about the logic below; the three labelled terms are what to hold onto.

The batch shift is **rank-1**: one scalar per slide, `gamma_a`, times one scalar per lipid, `lambda_c`.
That factorization is the modeling assumption that does all the work. It says every slide has a single
"how bad is my drift" number, and every lipid has a single "how much do I respond to drift" number, and
the shift on a given (slide, lipid) is just their product. The paper justifies this empirically: take
the matrix of measured technical errors and do an SVD, and the first singular value dominates, meaning
one slide-factor times one lipid-factor explains most of the technical variance.

`delta_c` is the systematic, biological distance between the background and foreground modes, the same
across slides for a given lipid. So the foreground mean is the background anchor, plus a biological gap
that should not be touched, plus a technical shift that should be removed. Normalization removes
`gamma_a * lambda_c` and leaves `delta_c` and the anchor in place. The foreground standard deviation is
built **additively**, `sigma1 = sigma_s_a + sigma_v_c`, a per-slide piece plus a per-lipid piece; this
is an implementation detail, the rank-1 batch shift is the idea to remember.

### how it is fit: MAP via SVI, not posterior sampling

uMAIA fits this model with **stochastic variational inference (SVI)** using an `AutoDelta` guide. That
phrase has a simple meaning. A delta-function guide collapses the "posterior" to a single point, so SVI
here is doing **maximum-a-posteriori (MAP)** estimation: it finds the single most probable set of
parameter values, with the priors acting as gentle regularizers, by gradient descent (Adam). There is
**no MCMC and there are no posterior samples.** Think "regularized best-fit point estimate", not
"Bayesian sampling". The discrete per-pixel labels `z` are marginalized analytically by enumeration, so
the optimizer only ever sees continuous parameters. With a fixed seed the result is deterministic.

The three calls, mirroring `scripts/run_umaia.py`:

1. `uMAIA.norm.initialize(x, mask, subsample=True)` does the **GMM initialization**: per molecule, fit
   a one- and a two-component Gaussian mixture and keep the lower-BIC one. This is where the background
   and foreground modes the model starts from come from.
2. `uMAIA.norm.normalize(...)` is the only heavy step: the **MAP fit via SVI**, `num_steps=2000`,
   `seed=42`, `covariate_vector=None` (more on that choice below). It subsamples about 2,500 pixels per
   section, so on a CPU it finishes in a couple of minutes.
3. `uMAIA.norm.transform(x, mask, params)` applies the **histogram-matching correction** (the
   CDF/inverse-CDF map we unroll by hand further down) and returns the normalized log tensor.

## building uMAIA's input tensor from our AnnData

uMAIA normalizes the **same molecule across sections**, so its input is not a flat pixels-by-lipids
matrix; it is a three-dimensional tensor `x` of shape `(N_pixels, S_sections, V_molecules)`, plus a
boolean `mask` marking which pixel slots are real (the two sections have different pixel counts, so the
shorter one is padded). For each section we take that section's pixels, log-transform the raw
intensities with the same `eps = 2e-4` offset, and stack them along the section axis. This mirrors
`scripts/build_umaia_input.py` exactly, except we read the ions straight from the AnnData you already
loaded instead of re-pulling them from METASPACE.

🔬 **TASK.** Build the tensor, then run the three uMAIA calls. The fit will print a tqdm progress bar
with a falling loss; that is SVI walking the parameters downhill toward their most probable values. The loss it reports is the negative ELBO, the objective variational inference optimizes; with the point-estimate (`AutoDelta`) guide used here, minimizing it is just gradient descent to the MAP fit. Let the bar fill (a couple of minutes).

💡 **HINT: read the code you are about to run.** Before you press shift-enter on the fit, open and read **`scripts/run_umaia.py`**: it is the exact three-call recipe (`initialize` -> `normalize` -> `transform`) you are about to reproduce inline, plus the parameter-saving (`uMAIA.ut.tools.save_svi`) and the same before/after money plot. The tensor builder it feeds from is **`scripts/build_umaia_input.py`**; open that too and check the `eps = 2e-4` log offset and the `(N, S, V)` padding match what the next cell does. You will never be asked to run a black box in this course without first being shown the box.

In [ ]:
# --- build the (N_pixels, S_sections, V_molecules) tensor from adata.X ---
molecules = list(adata.var_names)                       # the formula__adduct keys
sections = ["naive", "pregnant"]                         # section axis order (s=0 naive, s=1 pregnant)
cond = adata.obs["Condition"].to_numpy()
per_section_masks = [cond == s for s in sections]
counts = [int(m.sum()) for m in per_section_masks]       # pixels in each section

N = max(counts); S = len(sections); V = len(molecules)
x = np.zeros((N, S, V), np.float32)                      # log-intensities, padded
mask = np.zeros((N, S, V), bool)                         # True where a pixel is real
for s, m in enumerate(per_section_masks):
    xs = np.log(np.clip(X[m], 0, None) + eps)            # this section: log raw intensities
    n = xs.shape[0]
    x[:n, s, :] = xs
    mask[:n, s, :] = True
print(f"tensor x {x.shape} (N, S, V) | {V} molecules | sections {sections} | pixels {counts}")

# --- the REAL uMAIA fit, exactly as in scripts/run_umaia.py ---
print("initialize (GMM per molecule)...")
init = uMAIA.norm.initialize(x, mask, subsample=True)
print("normalize (MAP via SVI, 2000 steps)...")
svi = uMAIA.norm.normalize(x, mask, init_state=init, subsample=True, num_steps=2000, seed=RNG_SEED)
print("transform (histogram matching)...")
x_maia = np.asarray(uMAIA.norm.transform(x, mask, svi))  # normalized log tensor, same shape as x
print("done. x_maia", x_maia.shape)

💡 **HINT.** Two things to notice about the fit you just ran. It subsampled to about 2,500 pixels
per section, which is why it converged in a couple of minutes on the CPU even though each section has
tens of thousands of tissue pixels. And `covariate_vector` was left at its default `None`: we did *not*
hand the model the biological labels. The reason is in the corner-case section below; for now, trust
that telling the normalizer "this section is pregnant" would let it absorb the very biology we want to
test. uMAIA can also save its fitted parameters to disk with `uMAIA.ut.tools.save_svi(svi, "some_dir")`
and `transform` will read them back from that directory; passing the `svi` object directly, as we do, is
the in-memory shortcut.

🔬 **TASK.** Now look at what the fit did. For a handful of molecules, overlay the two sections'
per-molecule log histograms **before** (raw, top row) and **after** (uMAIA-normalized, bottom row).
Watch the two raw distributions, misaligned by the slide-to-slide offset, get pulled onto each other.

In [ ]:
# before/after per-section histograms for a few molecules (this is the money plot)
# 🔬 your code here


**check.** Top row: the two raw histograms for each molecule sit at visibly different places, the
slide-to-slide offset we have been warning about. Bottom row: after uMAIA, the same two distributions
overlap much better. The correction did not collapse them onto a single spike; it kept each section's
shape and slid them into register, which is exactly the conservative, order-preserving quantile matching
we will unroll by hand below. The alignment is not perfect, and with only two sections it should not be:
uMAIA cannot fully separate a genuine difference from a batch effect, so it leaves some real biology in
place rather than over-correcting.

🔬 **TASK.** Put one number on it. For every molecule, measure the gap between the two sections at
the **90th percentile** (a robust stand-in for where the foreground mode sits), then take the median of
that gap over all 104 molecules, before and after.

In [ ]:
# quantify the cross-section offset: median over molecules of the 90th-percentile gap
# 🔬 your code here


**check.** The median 90th-percentile gap between the two sections drops from about **0.21** log
units on the raw data to about **0.15** after uMAIA, a cut of roughly a quarter to a third, across all 104
molecules at once. That is the batch offset being removed. Your exact figures will wobble by a hundredth
or two from run to run, because uMAIA subsamples about 2,500 pixels per section for the fit and that draw
is not fully seeded; the direction (a clear shrinkage) is the stable, reproducible part. The residual gap
is not zero, and it should not be: with only two sections uMAIA cannot tell a genuine
control-versus-pregnant difference from a batch effect, so it leaves real biology in place rather than
over-correcting. We unpack that trade-off in the corner-case section.

## brief excursus: computing a compound score from features: the sphingolipids signal

Sphingolipids (the `HexCer`, `Cer` and `SM` classes) are concentrated in **myelin**, so
they light up the **white matter**: the corpus callosum, the fimbria, the internal capsule, the optic
tracts. Both our sections are normal adult brains cut at the same plane, so the white-matter
sphingolipid level should agree across them. This is better done after uMAIA normalization.

We build a simple **composite sphingolipid score**: z-score each sphingolipid across pixels (so each
contributes on a common scale), then average. We compute it on the raw data and on the
uMAIA-normalized data, and compare the white-matter mean between the two sections.

In [ ]:
# reconstruct the per-pixel uMAIA values on the native (un-logged) scale, in adata's row order.
# adata is ordered naive-block then pregnant-block, so we stack the two sections the same way.
nC, nP = counts
umaia_native = np.vstack([np.exp(x_maia[:nC, 0, :]),     # naive pixels, section axis 0
                          np.exp(x_maia[:nP, 1, :])]      # pregnant pixels, section axis 1
                         ).astype(np.float32)
assert umaia_native.shape == X.shape

# sphingolipid columns + a white-matter pixel mask from the provided Allen acronyms
sph_cols = [i for i, n in enumerate(adata.var["lipid"]) if n.startswith(("HexCer", "Cer", "SM"))]
WM = {"cc", "fi", "int", "or", "ec", "alv", "fp", "df", "st", "opt", "ml", "py",
      "cpd", "arb", "em", "fx", "ccg", "scwm"}
wm = adata.obs["acronym"].astype(str).isin(WM).to_numpy()
print(f"{len(sph_cols)} sphingolipid ions | {wm.sum():,} white-matter pixels ({wm.mean():.0%})")

def sphingo_score(arr):                                   # z-score each sphingolipid, then average
    z = (arr - arr.mean(0)) / (arr.std(0) + 1e-9)
    return z[:, sph_cols].mean(1)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
sc = sphingo_score(umaia_native)
for ax, c in zip(axes, sections):
    m = (cond == c)
    sm = ax.scatter(adata.obs.loc[m, "zccf"], -adata.obs.loc[m, "yccf"], c=sc[m],
                    cmap=CMAP_SEQUENTIAL, vmin=-0.5, vmax=2.0, s=2, rasterized=True)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title(f"{c}: sphingolipid score (uMAIA)", fontsize=FS["m"])
fig.colorbar(sm, ax=axes, shrink=0.6, label="composite z")
plt.show()

for label, arr in [("raw", X), ("uMAIA", umaia_native)]:
    s = sphingo_score(arr)
    nv = s[(cond == "naive") & wm].mean(); pg = s[(cond == "pregnant") & wm].mean()
    print(f"{label:6s}: white-matter sphingo z   naive {nv:+.3f}   pregnant {pg:+.3f}   gap {abs(nv - pg):.3f}")

## unrolling the correction: histogram matching by CDF and inverse-CDF

You called `uMAIA.norm.transform(...)` and watched the histograms align. Now we open that black box. The
transform is about twenty lines of NumPy and SciPy, and once you see it you will never find quantile
matching mysterious again. uMAIA does exactly this, only fit **jointly across all molecules at once**
with the modes learned from the MAP fit, whereas below we do it one molecule at a time so every step is
visible.

The idea. Suppose the measured log-intensity distribution of a lipid in one section has cumulative
distribution function $G(x)$, and we have a **reference** distribution with CDF $F$ that we want every
section to match. For any measured value $x$, $G(x)$ is its **quantile**: the fraction of pixels below
it, a number in $[0, 1]$. That quantile is the part we trust, because it encodes a pixel's *rank* within
its own section, which the batch effect does not change. To move the value onto the reference scale we
ask: what value in the reference distribution sits at that same quantile? That is $F^{-1}$, the inverse
CDF. So the whole correction is one composition:

$$x \;\longmapsto\; F^{-1}\big(G(x)\big).$$

Read it left to right: take a value, find its quantile under its own section's fitted CDF $G$, then read
off the value at that quantile under the shared reference $F$. Because it preserves quantiles, it is
**monotone**: it never reorders pixels within a section, it only restretches the axis. And because $G$
and $F$ are two-Gaussian mixtures, the stretch can be different in the foreground than in the
background, which is exactly what lets the correction hold the background anchor while sliding the
foreground mode. Let us build the two pieces.

💡 **HINT: read the code you are about to unroll.** The next cell rebuilds uMAIA's transform from scratch, but you should see the original first. Open **`uMAIA/normalization/_transform.py`** in your environment (`python -c "import uMAIA, os; print(os.path.dirname(uMAIA.__file__))"` prints the package root) and find `cdf_mixture` and `make_inv_cdf`: our three helpers below are line-for-line the same idea, only applied one molecule at a time so every step is visible. Reading uMAIA's version next to ours is the fastest way to convince yourself the `transform` call you already ran is exactly this $F^{-1}(G(x))$ map and nothing more.

In [ ]:
# --- the two transparent pieces of uMAIA's histogram-matching correction ---
# (these mirror cdf_mixture / make_inv_cdf in uMAIA/normalization/_transform.py)

def cdf_mixture(x, mu1, mu2, sigma1, sigma2):
    # Equal-weight two-Gaussian mixture CDF: the average of two normal CDFs.
    # mu1/sigma1 describe the background mode, mu2/sigma2 the foreground mode.
    # G(x) returns the quantile of x under this fitted mixture.
    return (stats.norm.cdf(x, mu1, sigma1) + stats.norm.cdf(x, mu2, sigma2)) / 2.0

def make_inv_cdf(mu1, mu2, sigma1, sigma2, resolution=2000, rel_range=6):
    # Build F^{-1} for a reference mixture by tabulating its CDF and inverting it.
    # Lay a fine grid over the value axis (+/- 6 sigma around the two modes),
    # compute the mixture CDF on that grid, then interpolate the OTHER way:
    # from cdf value (a quantile) back to the value. interp1d gives us F^{-1}.
    mu_min, mu_max = min(mu1, mu2), max(mu1, mu2)
    sigma_max = max(sigma1, sigma2)
    domain = np.linspace(mu_min - rel_range * sigma_max,
                         mu_max + rel_range * sigma_max, resolution)
    cdf_vals = cdf_mixture(domain, mu1, mu2, sigma1, sigma2)
    return interp1d(cdf_vals, domain, bounds_error=False, fill_value="extrapolate", copy=True)

def transform_byicdf(x, mu1x, mu2x, sigma1x, sigma2x, icdf_ref):
    # The F^{-1}(G(x)) map: quantile under the source CDF, value under the reference.
    return icdf_ref(cdf_mixture(x, mu1x, mu2x, sigma1x, sigma2x))

print("built cdf_mixture, make_inv_cdf, transform_byicdf")

💡 **HINT.** `interp1d(cdf_vals, domain)` looks backwards at first glance, and that is exactly the
trick. A normal CDF maps value to quantile. By handing `interp1d` the quantiles as the x-argument and
the values as the y-argument, we get a function that maps quantile back to value, which is $F^{-1}$. The
`make_inv_cdf` in uMAIA does precisely this, on a 2000-point grid.

### a controlled demo: two mismatched histograms snapping into register

To *see* the mechanism cleanly, we first build a controlled example: two simulated sections of the same
lipid, each a background hump plus a foreground hump, where section B's foreground has been shifted and
stretched by a fake batch effect. This is exactly the situation on raw data, but with the truth known.
Then we apply the CDF/inverse-CDF map to push both onto a shared reference and watch the foreground modes
line up.

🔬 **TASK.** Simulate the two mismatched histograms.

In [ ]:
# --- simulate two sections of one lipid: a bg mode + a fg mode each, in LOG space ---
# 🔬 your code here


**check.** The two background humps sit on top of each other near -8 (the stable anchor), but
section B's foreground hump is shoved right to -4 and is wider, while section A's sits at -5. That gap is
the batch effect we want gone.

🔬 **TASK.** Now build the **shared reference** the way uMAIA does (paper Eq 7): the reference mixture
parameters are the **mean over sections** of the fitted per-section parameters. With two sections that is
just the midpoint. Then map each section through $F_{ref}^{-1}(G_{section}(x))$.

In [ ]:
# --- shared reference = mean of the two sections' fitted mixture parameters (Eq 7) ---
# 🔬 your code here


**check.** After the transform, both foreground modes land on the dashed reference line at -4.5,
the midpoint, and the printed medians confirm it. The two distributions now overlap. We did not subtract
a constant or divide by a scalar; we matched quantiles through fitted CDFs, which lets the correction be
different in the foreground than in the background. The background anchor barely moved, the foreground
modes converged, and within each section every pixel kept its rank.

❓ **QUESTION.** The reference is the *average* of the two sections, so both move toward each other rather
than one snapping to the other. With only two sections, what happens to that reference if one slide is a
technical outlier? Hold that thought; it is the heart of the corner case below.

## a second, separate normalization: per-lipid 0-to-1 scaling

uMAIA made the same *lipid* comparable across *sections*. There is a second, independent problem:
different *lipids* are not comparable to each other on the raw intensity scale. The reason is physical.
MALDI is only **semi-quantitative**: a lipid's measured intensity depends on how easily it ionizes,
which varies wildly from one molecule to another. A lipid that ionizes efficiently reads bright even at
modest abundance; a poor ionizer reads dim even when it is abundant. So you cannot read "lipid X is 0.08
and lipid Y is 0.20, therefore there is more Y". The numbers live on different, lipid-specific scales.

The fix is to put every lipid on a common yardstick: clip the extreme tails and rescale each lipid into
[0, 1], so a value of 0.7 means "high *for this lipid*" regardless of which lipid it is. That is the only
sense in which MALDI intensities compare across molecules, and it is what every cross-lipid step later in
the course runs on. It is a different operation from uMAIA, applied later (in the embedding notebook),
and we unroll it here so you have seen it.

💡 **HINT.** The unrolled cell below is identical to **`cl.analysis.min01_per_lipid`**; open `src/cajal_lipidomics/analysis.py`, read that function, and confirm the clip-and-rescale you write by hand is exactly what the helper does before you trust the `assert` that checks it.

In [ ]:
# --- per-lipid 0-1 normalization, unrolled (this is exactly cl.analysis.min01_per_lipid) ---
# we run it on the uMAIA-normalized values, the scale everything downstream uses
Xn = umaia_native.astype(float)
lo, hi = 0.005, 0.995                              # clip the extreme 0.5% tails per lipid
ql = np.quantile(Xn, lo, axis=0)                   # per-lipid low clip (one value per column)
qh = np.quantile(Xn, hi, axis=0)                   # per-lipid high clip
span = np.where(qh > ql, qh - ql, 1.0)             # guard against a flat (constant) lipid
X01 = np.clip((Xn - ql) / span, 0, 1)              # rescale each lipid into [0, 1]

# the helper does exactly the same thing; verify they agree
X01_helper = analysis.min01_per_lipid(umaia_native, lo=0.005, hi=0.995)
print("max abs difference between unrolled and helper:", float(np.max(np.abs(X01 - X01_helper))))
assert np.allclose(X01, X01_helper), "helper and unrolled version should match"
print("they match: cl.analysis.min01_per_lipid is the unrolled recipe above")

In [ ]:
# see the scale collapse: three lipids before (very different ranges) and after ([0,1])
show3 = ["HexCer 42:2", "PC 36:4", "PA 34:1"]
cols = [list(adata.var["lipid"]).index(s) for s in show3]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for c, name in zip(cols, show3):
    axes[0].hist(Xn[:, c], bins=80, histtype="step", lw=1.4, label=name)
    axes[1].hist(X01[:, c], bins=80, histtype="step", lw=1.4, label=name)
axes[0].set_title("before: each lipid on its own scale", fontsize=FS["m"])
axes[0].set_xlabel("uMAIA intensity"); axes[0].set_ylabel("pixels"); axes[0].legend(fontsize=FS["s"], frameon=False)
axes[1].set_title("after: every lipid on a common [0, 1]", fontsize=FS["m"])
axes[1].set_xlabel("per-lipid 0-1 value"); axes[1].legend(fontsize=FS["s"], frameon=False)
plt.tight_layout(); plt.show()

**check.** Before, the three lipids live on visibly different ranges, so a raw cross-lipid
comparison would be meaningless. After, all three fill the same [0, 1] axis, and 0.7 means "high for this
lipid" no matter which lipid it is.

❓ **QUESTION.** Per-lipid 0-1 scaling and uMAIA normalization solve *different* problems. State each in
one sentence. (uMAIA: make the same lipid comparable across sections. min01: make different lipids
comparable to each other.) Both are necessary; neither replaces the other.

## saving your output for the next notebook

This is your N3 deliverable: the two sections, made comparable. We store the uMAIA-normalized values as
a **layer** on the AnnData (`adata.layers["umaia"]`, on the native scale) and keep the raw `X` untouched,
then write `data/derived/03_normalized.h5ad`. The next notebook loads exactly this file and builds the
embedding on the uMAIA layer. Each notebook consumes the previous stage, so the course runs as one
sequential pipeline.

🔬 **TASK.** Attach the layer and write the file.

In [ ]:
# attach the uMAIA-normalized layer (native scale, same row order as adata) and save
# 🔬 your code here


**check.** `data/derived/03_normalized.h5ad` is written, with `layers` now listing `umaia`, the raw
`X` still on its 0-to-1 scale, and the `umaia` layer on its native (un-logged) scale. That file is the
input to N4.

## what you did, in one breath

You met the **batch effect** and saw why it is dangerous: the same lipid reads a different raw intensity
on different slides for purely technical reasons, so a naive control-versus-pregnant comparison would
partly measure the slide. You saw the empirical fact uMAIA stands on, the **bimodal log histogram** with
a stable background anchor and a drifting foreground mode. You read the **model**: a two-Gaussian mixture
per (lipid, section) whose foreground mean carries a **rank-1 batch shift** `gamma_a * lambda_c`, fit by
**MAP via SVI**, not by sampling. You built uMAIA's input tensor from your own annotated AnnData and
**ran the real fit**, watched the median cross-section offset shrink. You unrolled the
**correction** by hand, pushing each value through its own CDF to a quantile and back through a shared
reference inverse-CDF, in simulation and on a real lipid. You faced the **two-section corner case**
honestly. And you applied the second, separate normalization, **per-lipid 0-to-1 scaling**.

Two rules to carry forward:

- **uMAIA makes the same lipid comparable across sections; min01 makes different lipids comparable to
  each other.** Different problems, both necessary.
- **Differential testing runs on the uMAIA-normalized, non-Harmonized data.** Never test on data whose
  batch correction also touched the biology.

Next, in N4, we use this comparable data to build the embedding and start finding the lipid territories
called lipizones.